# Milan Airbnb - Data Preparation Pipeline

This notebook performs the complete ETL pipeline for Milan Airbnb data:
1. Load raw data from Inside Airbnb
2. Clean listings, calendar, and reviews
3. Create spatial data structures
4. Spatial join listings to neighbourhoods
5. Aggregate metrics to neighbourhood level
6. Quality checks and visualization

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

from src.config import (
    INPUT_FILES, OUTPUT_FILES, CRS_WEB, CRS_METRIC,
    LISTINGS_SAMPLE_SIZE, RANDOM_SEED, CITY_NAME
)
from src.io import load_csv, load_geojson, save_parquet, save_geojson
from src.cleaning import clean_calendar, clean_listings
from src.reviews import clean_and_aggregate_reviews
from src.spatial import (
    listings_to_geodataframe, clean_neighbourhoods,
    spatial_join_listings_neighbourhoods, aggregate_to_neighbourhoods
)
from src.qc import (
    check_unique_ids, check_geometry_validity, check_crs,
    check_price_range, print_qc_report
)

## Step 1: Load Raw Data

In [ ]:
listings_raw = load_csv(INPUT_FILES['listings_detailed'])
calendar_raw = load_csv(INPUT_FILES['calendar_detailed'])
reviews_raw = load_csv(INPUT_FILES['reviews_detailed'])
neighbourhoods_raw = load_geojson(INPUT_FILES['neighbourhoods_geojson'])

print(f"Listings: {len(listings_raw)} rows")
print(f"Calendar: {len(calendar_raw)} rows")
print(f"Reviews: {len(reviews_raw)} rows")
print(f"Neighbourhoods: {len(neighbourhoods_raw)} polygons")

## Step 2: Clean Calendar

In [ ]:
calendar_clean = clean_calendar(calendar_raw)
save_parquet(calendar_clean, OUTPUT_FILES['calendar_clean'])
print(f"Calendar cleaned and saved: {len(calendar_clean)} rows")

## Step 3: Clean Listings

In [ ]:
listings_clean = clean_listings(listings_raw)
save_parquet(listings_clean, OUTPUT_FILES['listings_clean'])
print(f"Listings cleaned and saved: {len(listings_clean)} rows")
listings_clean.head()

## Step 4: Clean and Aggregate Reviews

In [ ]:
reviews_agg = clean_and_aggregate_reviews(reviews_raw)
save_parquet(reviews_agg, OUTPUT_FILES['reviews_listing_features'])
print(f"Reviews aggregated: {len(reviews_agg)} listings")

## Step 5: Create Spatial Data

In [ ]:
gdf_listings = listings_to_geodataframe(listings_clean)
gdf_neighbourhoods = clean_neighbourhoods(neighbourhoods_raw)

print(f"Listings GeoDataFrame: {len(gdf_listings)} points")
print(f"Neighbourhoods GeoDataFrame: {len(gdf_neighbourhoods)} polygons")

## Step 6: Spatial Join

In [ ]:
gdf_joined = spatial_join_listings_neighbourhoods(gdf_listings, gdf_neighbourhoods)
print(f"Joined listings: {len(gdf_joined)}")

## Step 7: Aggregate to Neighbourhood Level

In [ ]:
neighbourhoods_enriched = aggregate_to_neighbourhoods(gdf_listings, gdf_neighbourhoods)
save_parquet(neighbourhoods_enriched, OUTPUT_FILES['neighbourhoods_enriched'])
save_geojson(neighbourhoods_enriched, OUTPUT_FILES['neighbourhoods_enriched_geojson'])
print(f"Neighbourhoods enriched: {len(neighbourhoods_enriched)}")
neighbourhoods_enriched.head()

## Step 8: Create Sample for Webmap

In [ ]:
sample = gdf_joined.sample(n=min(LISTINGS_SAMPLE_SIZE, len(gdf_joined)), random_state=RANDOM_SEED)
save_geojson(sample, OUTPUT_FILES['listings_points_enriched_sample'])
print(f"Sample created: {len(sample)} listings")

## Step 9: Quality Checks

In [ ]:
print("=" * 60)
print("QUALITY CHECKS")
print("=" * 60)

print("\n--- Listings ---")
check_unique_ids(listings_clean, 'listing_id')
check_price_range(listings_clean, 'price')

print("\n--- Listings GeoDataFrame ---")
check_geometry_validity(gdf_listings)
check_crs(gdf_listings, CRS_WEB)

print("\n--- Neighbourhoods ---")
check_geometry_validity(gdf_neighbourhoods)
check_crs(gdf_neighbourhoods, CRS_WEB)

print("\n" + "=" * 60)

## Step 10: Data Quality Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(listings_clean['price'], bins=50, ax=axes[0, 0], kde=True)
axes[0, 0].set_title('Price Distribution')
axes[0, 0].set_xlabel('Price (EUR)')

if 'accommodates' in listings_clean.columns:
    sns.histplot(listings_clean['accommodates'], bins=20, ax=axes[0, 1])
    axes[0, 1].set_title('Accommodates Distribution')

if 'room_type' in listings_clean.columns:
    listings_clean['room_type'].value_counts().plot(kind='bar', ax=axes[1, 0])
    axes[1, 0].set_title('Room Type Distribution')
    axes[1, 0].tick_params(axis='x', rotation=45)

if 'neighbourhood_cleansed' in listings_clean.columns:
    top_neigh = listings_clean['neighbourhood_cleansed'].value_counts().head(15)
    top_neigh.plot(kind='barh', ax=axes[1, 1])
    axes[1, 1].set_title('Top 15 Neighbourhoods by Listing Count')

plt.tight_layout()
plt.savefig(project_root / 'reports' / 'figures' / '01_data_quality_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11: Spatial Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

gdf_neighbourhoods.plot(ax=ax, facecolor='lightgray', edgecolor='gray', alpha=0.5)

if 'price' in gdf_listings.columns:
    gdf_listings.plot(
        ax=ax,
        column='price',
        cmap='YlOrRd',
        markersize=5,
        alpha=0.6,
        legend=True,
        legend_kwds={'label': 'Price (EUR)', 'orientation': 'vertical'}
    )

ax.set_title(f'{CITY_NAME} Airbnb Listings by Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.savefig(project_root / 'reports' / 'figures' / f'fig_{CITY_NAME.lower()}_listings_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12: Neighbourhood Price Map

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

neighbourhoods_enriched.plot(
    ax=ax,
    column='median_price',
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Median Price (EUR)', 'orientation': 'vertical'},
    edgecolor='gray'
)

ax.set_title(f'{CITY_NAME} Median Airbnb Price by Neighbourhood', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.savefig(project_root / 'reports' / 'figures' / f'fig_{CITY_NAME.lower()}_neighbourhood_median_price.png', dpi=150, bbox_inches='tight')
plt.show()

## Pipeline Complete

The data preparation is complete. Next steps:
1. Run `scripts/01_verify_spatial_data.py` to verify processed data
2. Run `scripts/03_ols_price_analysis.py` for OLS regression
3. Run `scripts/04_spatial_autocorr_morans_i.py` for Moran's I
4. Run `scripts/05_lm_diagnostic_tests.py` for LM diagnostics
5. Run `scripts/07_spatial_models_sar_sem.py` for spatial models